# Test NLLB_200
AutoModelForSeq2SeqLM and AutoTokenizer are classes from Hugging Face's Transformers library.
- AutoTokenizer: Converts raw text → numbers (tokens) that the model understands.
- AutoModelForSeq2SeqLM: Loads the actual neural network model for seq2seq tasks.

In [ ]:
!pip install transformers sentencepiece pdfplumber -q

In [ ]:
# Fix: load model and tokenizer directly (skip pipeline)
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

print("Loading NLLB-200...")

model_name = "facebook/nllb-200-distilled-600M"
nllb_tokenizer  = AutoTokenizer.from_pretrained(model_name)
nllb_model  = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model loaded!")

Loading NLLB-200...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded!


In [ ]:
# If PDF, extract text first
import pdfplumber

with pdfplumber.open("/content/drive/MyDrive/AI-application-in-Law-Hackathon/data/VN_sample.pdf") as pdf:
    vn_text = "\n".join(page.extract_text() for page in pdf.pages if page.extract_text())

sample_text = vn_text[:1000]
print("=== VIETNAMESE ORIGINAL (first 1000 chars) ===")
print(sample_text)

=== VIETNAMESE ORIGINAL (first 1000 chars) ===
CÔNG BÁO/Số 971 + 972/Ngày 24-7-2025 13
QUỐC HỘI CỘNG HÒA XÃ HỘI CHỦ NGHĨA VIỆT NAM
Độc lập - Tự do - Hạnh phúc
Luật số: 91/2025/QH15
LUẬT
BẢO VỆ DỮ LIỆU CÁ NHÂN
Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam đã được sửa đổi,
bổ sung một số điều theo Nghị quyết số 203/2025/QH15;
Quốc hội ban hành Luật Bảo vệ dữ liệu cá nhân.
Chương I
NHỮNG QUY ĐỊNH CHUNG
Điều 1. Phạm vi điều chỉnh và đối tượng áp dụng
1. Luật này quy định về dữ liệu cá nhân, bảo vệ dữ liệu cá nhân và quyền,
nghĩa vụ, trách nhiệm của cơ quan, tổ chức, cá nhân có liên quan.
2. Luật này áp dụng đối với:
a) Cơ quan, tổ chức, cá nhân Việt Nam;
b) Cơ quan, tổ chức, cá nhân nước ngoài tại Việt Nam;
c) Cơ quan, tổ chức, cá nhân nước ngoài trực tiếp tham gia hoặc có liên quan
đến hoạt động xử lý dữ liệu cá nhân của công dân Việt Nam và người gốc Việt
Nam chưa xác định được quốc tịch đang sinh sống tại Việt Nam đã được cấp giấy
chứng nhận căn cước.
Điều 2. Giải thích từ ng

In [ ]:
# Translate function
def translate_vn_to_en_nllb(text):
    inputs = nllb_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    translated = nllb_model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids("eng_Latn"),
        max_length=512
    )

    return nllb_tokenizer.decode(translated[0], skip_special_tokens=True)

# Test with your sample
result = translate_vn_to_en_nllb(sample_text)
print("=== ENGLISH TRANSLATION ===")
print(result)

=== ENGLISH TRANSLATION ===
NOTE 971 + 972/Day 24-7-2025 13 QUỐC HI CNG HI CNG HI HI HI VIT NAM Independence - Freedom - Happiness Law No: 91/2025/QH15 LUT BO V D LIU CÁ NHÂN Basic Law of the Socialist Republic of Vietnam has been amended, adding some direct provisions not in accordance with resolution 203/2025/QH15; the National Assembly has banned the Law on the Protection of Personal Data. Chapter I CHNG QUYNHUNG Article 1. The Law has amended and the subject has applied 1. This Law provides for the processing of personal data, the protection of personal data and, in the sense of the law, the responsibility of the institution, the language of the person concerned. 2. This Law defines the legal status of the person, the legal status of the person, the legal status of the person, the legal status of the person, the legal status of the person, the legal status of the person, the legal status of the person, the legal status of the person, the legal status of the person, the legal status